In [31]:
from IPython import get_ipython
ip = get_ipython()
ip.run_line_magic("reload_ext", "autoreload")  # these will enable module autoreloading
ip.run_line_magic("autoreload", "2")

### Load in Exports

In [32]:
import os
from lxml import etree
import BrokerageExportProviders.Brokerages.IBKR.IbkrBrokerageExportProvider as ibrk
import ConfigurationProvider.Configuration as conf
import Core.StagingFinancialEvents.Services.StagingFinancialGroupingProcessor as sgp
import Core.StagingFinancialEvents.Utils.ProcessingUtils as pu
import arrow
import TaxAuthorityProvider.TaxAuthorities.Slovenia.SlovenianTaxAuthorityProvider as tap
import TaxAuthorityProvider.TaxAuthorities.Slovenia.Schemas.ReportTypes as rt
import TaxAuthorityProvider.Schemas.Configuration as cf

rootOfProject = os.getcwd()

reportsDirectory = os.path.join(rootOfProject, "imports")
generatedDirectory = os.path.join(rootOfProject, "exports")
configDirectory = os.path.join(rootOfProject, "config")

configProvider = conf.ConfigurationProvider(configDirectory)
taxPayerInfo = configProvider.getConfig()

ibkrProvider = ibrk.IbkrBrokerageExportProvider()
brokerExports = ibkrProvider.getListOfReportsAvailableForBroker(reportsDirectory)
print(brokerExports)



def loadFileAndExtractLines(file: str):  
    transactions = ibkrProvider.getBrokerEventsForReport(file)
    return transactions

brokerReports = list(map(loadFileAndExtractLines, brokerExports))
mergedReports = ibkrProvider.mergeBrokerEvents(brokerReports)

['/mnt/extra/Projects/IB-tax-calculator/imports/SMCI.xml']


### Process Broker Exports

In [33]:
convertedCommonFormat = ibkrProvider.transformBrokerEventsToBrokerAgnosticEvents(mergedReports)
groupingProcessor = sgp.StagingFinancialGroupingProcessor(pu.ProcessingUtils())

financialEvents = groupingProcessor.processStagingFinancialEvents(convertedCommonFormat)

### Events of Interest

In [34]:
import pandas as pd
from Core.FinancialEvents.Schemas.IdentifierRelationship import IdentifierChangeType
from Core.FinancialEvents.Services.ApplyIdentifierRelationshipsService import ApplyIdentifierRelationshipsService
from notebooks.utils.provenance_table import collectProvenanceTableForStocks, rowKeyFromRow

applyService = ApplyIdentifierRelationshipsService()
appliedEvents = applyService.apply(
    financialEvents,
    changeTypesToApply=[IdentifierChangeType.RENAME, IdentifierChangeType.SPLIT, IdentifierChangeType.REVERSE_SPLIT],
)

# Only include items relevant to the stock report
provenanceDf, affectedByKey = collectProvenanceTableForStocks(appliedEvents)
if provenanceDf.empty:
    print("No provenance events (no renames/splits applied).")
else:
    provenanceDf["AffectedCount"] = provenanceDf.apply(lambda r: len(affectedByKey.get(rowKeyFromRow(r), [])), axis=1)
    display(provenanceDf)

    print("\n--- Affected events per provenance step ---")
    for idx, r in provenanceDf.iterrows():
        key = rowKeyFromRow(r)
        items = affectedByKey.get(key, [])
        if not items:
            continue
        from_lbl = r["FromISIN"] or r["FromTicker"] or "?"
        to_lbl = r["ToISIN"] or r["ToTicker"] or "?"
        qty = f" ({r['QuantityBefore']} → {r['QuantityAfter']})" if pd.notna(r.get("QuantityBefore")) and r.get("QuantityBefore") is not None else ""
        print(f"\n{r['ChangeType']} {from_lbl} → {to_lbl} ({r['EffectiveDate']}){qty}: {len(items)} item(s)")
        affectedDf = pd.DataFrame([{"Type": it[0], "ID": it[1], "Date": it[2], "Quantity": it[3]} for it in items])
        display(affectedDf)

,ChangeType,EffectiveDate,FromISIN,FromTicker,ToISIN,ToTicker,QuantityBefore,QuantityAfter,AffectedCount
0,SPLIT,2024-09-30,US86800U1043,SMCI.OLD,US86800U3023,SMCI,4.0,40.0,5



--- Affected events per provenance step ---

SPLIT US86800U1043 → US86800U3023 (2024-09-30) (4.0 → 40.0): 5 item(s)


,Type,ID,Date,Quantity
0,Grouping,US86800U3023,None,NaN
1,StockTrade,787031269,2024-08-01,10.0
2,StockTrade,804731731,2024-08-27,10.0
3,StockTrade,809755805,2024-09-04,10.0
4,StockTrade,3120334679,2024-09-26,10.0


### Stock Reports

In [35]:
reportconfig = cf.TaxAuthorityConfiguration(fromDate=arrow.get("2025"), toDate=arrow.get("2026"), lotMatchingMethod=cf.TaxAuthorityLotMatchingMethod.FIFO)
# print(convertedCommonFormat)

provider = tap.SlovenianTaxAuthorityProvider(taxPayerInfo=taxPayerInfo, reportConfig=reportconfig)
tradeReport = provider.generateExportForTaxAuthority(reportType=rt.SlovenianTaxAuthorityReportTypes.DOH_KDVP, events=financialEvents)


tradeCsv = provider.generateSpreadsheetExport(reportType=rt.SlovenianTaxAuthorityReportTypes.DOH_KDVP, events=financialEvents)
tradeCsv.to_csv(os.path.join(generatedDirectory, 'export-trades.csv'), index=False)

etree.ElementTree(tradeReport).write(os.path.join(generatedDirectory, "Doh_KDVP_9.xml"), xml_declaration=True, encoding='utf-8', pretty_print=True)